In [ ]:
import subprocess

print("Verifying environment...")

timm_version = subprocess.check_output(['pip', 'show', 'timm'], text=True).split('\n')[1].split(': ')[1]
print(f"Current timm version: {timm_version}")

version_parts = timm_version.split('.')
major = int(version_parts[0])
minor = int(version_parts[1]) if len(version_parts) > 1 else 0

if major < 1 or (major == 1 and minor < 20):
    print("Upgrading timm to >= 1.0.20 for DINOv3 support...")
    subprocess.check_call(['pip', 'install', '--upgrade', 'timm'])
    print("timm upgraded successfully")
else:
    print(f"timm version {timm_version} is compatible")

import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: GPU not detected. Enable GPU in Runtime → Change runtime type")

In [ ]:
import sys
import os

IS_COLAB = 'google.colab' in sys.modules

if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive mounted at /content/drive")
    BASE_PATH = '/content/drive/MyDrive'
else:
    BASE_PATH = '.'
    print("Running locally")

print(f"Base path: {BASE_PATH}")

## SETUP INSTRUCTIONS FOR CLUSTER EXECUTION

**Required before running:**

### 1. Download and Place DINOv3 Weights
- Download from: https://drive.google.com/file/d/1gLJJ_sq8R8d9Jcr3to695kTlVBv0fZtI/view?usp=sharing
- Extract to: `dinov3/vitl16-sat493m/dinov3_vitl16_pretrain_sat493m-eadcf0ff.pth`
- Code will **fail if weights not found** (required for training)

### 2. Ensure LandDiscover-50K Dataset
- Place dataset at: `data/datasets/LandDiscover_50K/TR_Image/`
- Should contain: 51,846 satellite images (.jpg, .png, or .jpeg)
- Code will **fail if no images found** (required for training)

### 3. Run on GPU
- Enable GPU before execution (Runtime → Change runtime type → GPU for Colab)
- CPU training will be extremely slow

### 4. Expected Output
- Checkpoints saved at: `./checkpoints/`
- Key file for GSNet: `./checkpoints/self_supervised_backbone_final.pth`
- Training logs printed after each epoch


# Self-Supervised DINOv3 Fine-Tuning

Fine-tune DINOv3 (ViT-L/16) on LandDiscover-50K using self-supervised learning.

**Approach:** Momentum-based consistency + spatial patch contrastive learning
- Works with small batch sizes (1-2)
- Learns spatial relationships between patches
- Teacher-student framework for stability

**Evaluation:** Train GSNet for 5K iterations with varying self-supervised epochs, measure mIoU

## Setup: Import Libraries

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from torch.utils.data import DataLoader
from torchvision import transforms as T
from PIL import Image
import timm
import copy
from tqdm import tqdm
import cv2

print("All libraries imported successfully")
print(f"  numpy: {np.__version__}")
print(f"  torch: {torch.__version__}")
print(f"  timm: {timm.__version__}")


In [ ]:
ROOT_DATA = os.path.join(BASE_PATH, "data/datasets/LandDiscover_50K")
IMG_DIR = os.path.join(ROOT_DATA, "TR_Image")
GT_DIR = os.path.join(ROOT_DATA, "GT_ID")
PRETRAINED_WEIGHTS = os.path.join(BASE_PATH, "dinov3/vitl16-sat493m/dinov3_vitl16_pretrain_sat493m-eadcf0ff.pth")

NUM_CLASSES = 40
IMG_SIZE = 384
BATCH_SIZE = 2
EPOCHS = 15
NUM_UNFROZEN_BLOCKS = 4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IGNORE_INDEX = 255

MOMENTUM = 0.99
TEMPERATURE = 0.07
SPATIAL_WEIGHT = 0.5
MOMENTUM_WEIGHT = 0.5

print(f"Paths configured:")
print(f"  Data root: {ROOT_DATA}")
print(f"  Images: {IMG_DIR}")
print(f"  Weights: {PRETRAINED_WEIGHTS}")
print(f"\nHyperparameters:")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Epochs: {EPOCHS}")
print(f"  Image size: {IMG_SIZE}")
print(f"  Device: {DEVICE}")

## Configuration

In [ ]:
print("\nChecking dataset availability...")

if not os.path.isdir(IMG_DIR):
    raise FileNotFoundError(f"Dataset NOT found at {IMG_DIR}\nPlace LandDiscover-50K images at: {IMG_DIR}")

num_images = len([f for f in os.listdir(IMG_DIR) if f.lower().endswith((".jpg", ".png", ".jpeg"))])
if num_images == 0:
    raise FileNotFoundError(f"No images found in {IMG_DIR}")

print(f"Dataset found: {num_images} images")
print(f"Location: {IMG_DIR}")


## Data Preparation: Real or Synthetic

In [ ]:
print("\nVerifying paths and weights...")

if not os.path.isdir(IMG_DIR):
    raise FileNotFoundError(f"Image directory not found: {IMG_DIR}")

print(f"Paths verified successfully")
print(f"  Images: {IMG_DIR} ({len(os.listdir(IMG_DIR))} files)")

if not os.path.isfile(PRETRAINED_WEIGHTS):
    raise FileNotFoundError(
        f"\nPretrained weights NOT found at:\n  {PRETRAINED_WEIGHTS}\n"
        f"Download from: https://drive.google.com/file/d/1gLJJ_sq8R8d9Jcr3to695kTlVBv0fZtI/view\n"
        f"Place at: {PRETRAINED_WEIGHTS}"
    )

weights_size = os.path.getsize(PRETRAINED_WEIGHTS) / 1e9
print(f"  Weights: {PRETRAINED_WEIGHTS} ({weights_size:.2f} GB)")


## Dataset: Images Only (No Labels)

In [ ]:
class LandDiscoverUnsupervisedDataset(Dataset):
    def __init__(self, img_root, transform=None):
        self.img_root = img_root
        self.transform = transform
        
        self.files = sorted([
            f for f in os.listdir(img_root)
            if f.lower().endswith((".jpg", ".png", ".jpeg"))
        ])
        
        assert len(self.files) > 0, "No images found in TR_Image"
    
    def __len__(self):
        return len(self.files)
    
    def __getitem__(self, idx):
        name = self.files[idx]
        img = Image.open(os.path.join(self.img_root, name)).convert("RGB")
        
        if self.transform:
            img1 = self.transform[0](img)
            img2 = self.transform[1](img)
            return img1, img2
        
        return img, img

print("Dataset class defined")

## Augmentation: Weak and Strong

In [ ]:
weak_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.5),
    T.ToTensor(),
    T.Normalize(mean=[0.5]*3, std=[0.5]*3),
])

strong_transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.5),
    T.RandomRotation(degrees=90),
    T.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1),
    T.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.8, 1.2)),
    T.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
    T.ToTensor(),
    T.Normalize(mean=[0.5]*3, std=[0.5]*3),
])

transforms = [weak_transform, strong_transform]
print("Augmentation pipelines defined")
print("  Weak: flip + rotate")
print("  Strong: flip + rotate + color jitter + affine + blur")

## DataLoader

In [ ]:
dataset = LandDiscoverUnsupervisedDataset(IMG_DIR, transform=transforms)
print(f"Dataset size: {len(dataset)}")

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

print(f"DataLoader created")
print(f"  Total batches per epoch: {len(loader)}")

## Load DINOv3 Backbone

In [ ]:
backbone = timm.create_model(
    "vit_large_patch16_dinov3.sat493m",
    pretrained=False,
    num_classes=0,
    dynamic_img_size=True,
    img_size=IMG_SIZE
).to(DEVICE)

print(f"Created backbone: ViT-L/16 DINOv3")
print(f"  Embedding dimension: {backbone.embed_dim}")
print(f"  Number of blocks: {len(backbone.blocks)}")
print(f"  Patch size: {backbone.patch_embed.patch_size}")

## Load Pretrained SAT-493M Weights

In [ ]:
print(f"Loading pretrained weights from: {PRETRAINED_WEIGHTS}")
ckpt = torch.load(PRETRAINED_WEIGHTS, map_location="cpu", weights_only=False)

if "teacher" in ckpt:
    backbone.load_state_dict(ckpt["teacher"], strict=False)
    print("Loaded from 'teacher' key")
elif "teacher_state_dict" in ckpt:
    backbone.load_state_dict(ckpt["teacher_state_dict"], strict=False)
    print("Loaded from 'teacher_state_dict' key")
elif "model" in ckpt:
    backbone.load_state_dict(ckpt["model"], strict=False)
    print("Loaded from 'model' key")
else:
    backbone.load_state_dict(ckpt, strict=False)
    print("Loaded raw state dict")

print("Pretrained weights loaded successfully")


## Freeze/Unfreeze Blocks

In [ ]:
for p in backbone.parameters():
    p.requires_grad = False

print(f"Unfreezing last {NUM_UNFROZEN_BLOCKS} blocks...")
for block in backbone.blocks[-NUM_UNFROZEN_BLOCKS:]:
    for p in block.parameters():
        p.requires_grad = True

total_params = sum(p.numel() for p in backbone.parameters())
trainable_params = sum(p.numel() for p in backbone.parameters() if p.requires_grad)
trainable_pct = 100 * trainable_params / total_params

print(f"Trainable parameters: {trainable_params:,} / {total_params:,} ({trainable_pct:.1f}%)")

## Teacher-Student Models (Momentum Encoder)

In [ ]:
student_model = backbone
teacher_model = copy.deepcopy(backbone)

for p in teacher_model.parameters():
    p.requires_grad = False

print("Teacher-student models created")
print(f"  Student: trainable")
print(f"  Teacher: frozen (momentum updates only)")

def update_teacher(student_model, teacher_model, tau=MOMENTUM):
    for student_param, teacher_param in zip(
        student_model.parameters(), 
        teacher_model.parameters()
    ):
        teacher_param.data = tau * teacher_param.data + (1 - tau) * student_param.data.detach()

print("Teacher update function defined")

## Self-Supervised Loss Functions

In [ ]:
class MomentumConsistencyLoss(nn.Module):
    def __init__(self, temperature=TEMPERATURE):
        super().__init__()
        self.temperature = temperature
    
    def forward(self, student_feat, teacher_feat):
        student_feat = F.normalize(student_feat, dim=-1, p=2)
        teacher_feat = F.normalize(teacher_feat, dim=-1, p=2)
        
        similarity = torch.sum(student_feat * teacher_feat, dim=-1)
        loss = 1 - similarity.mean()
        return loss

class SpatialPatchContrastiveLoss(nn.Module):
    def __init__(self, temperature=TEMPERATURE, spatial_radius=1):
        super().__init__()
        self.temperature = temperature
        self.spatial_radius = spatial_radius
    
    def forward(self, feat_map):
        B, C, H, W = feat_map.shape
        
        feat_map_norm = F.normalize(feat_map, dim=1, p=2)
        feat_flat = feat_map_norm.view(B, C, -1).transpose(1, 2)
        
        total_loss = 0.0
        count = 0
        
        for h in range(H):
            for w in range(W):
                center_feat = feat_flat[:, h*W + w, :]
                
                for dh in range(-self.spatial_radius, self.spatial_radius + 1):
                    for dw in range(-self.spatial_radius, self.spatial_radius + 1):
                        if dh == 0 and dw == 0:
                            continue
                        
                        nh, nw = h + dh, w + dw
                        if 0 <= nh < H and 0 <= nw < W:
                            neighbor_feat = feat_flat[:, nh*W + nw, :]
                            
                            similarity = torch.sum(center_feat * neighbor_feat, dim=-1) / self.temperature
                            loss = 1 - similarity.mean()
                            total_loss += loss
                            count += 1
        
        return total_loss / max(count, 1)

momentum_loss = MomentumConsistencyLoss().to(DEVICE)
spatial_loss = SpatialPatchContrastiveLoss().to(DEVICE)

print("Loss functions defined (FIXED):")
print("  1. Momentum Consistency Loss: 1 - cosine_similarity (positive values)")
print("  2. Spatial Patch Contrastive Loss: 1 - neighbor_similarity (positive values)")


## Optimizer

In [ ]:
backbone_lr = 5e-5
head_lr = 5e-4

optimizer = AdamW(
    [
        {"params": student_model.blocks[-NUM_UNFROZEN_BLOCKS:].parameters(), "lr": backbone_lr},
    ],
    weight_decay=1e-4
)

print(f"Optimizer configured:")
print(f"  Type: AdamW")
print(f"  Backbone LR: {backbone_lr}")
print(f"  Weight decay: 1e-4")

## Helper Functions

In [ ]:
def extract_patch_features(feat_tokens, img_size=IMG_SIZE, patch_size=16):
    B, N, C = feat_tokens.shape
    
    grid_size = img_size // patch_size
    expected_patches = grid_size * grid_size
    num_special_tokens = N - expected_patches
    
    spatial_tokens = feat_tokens[:, num_special_tokens:, :]
    
    feat_map = spatial_tokens.transpose(1, 2).reshape(
        B, C, grid_size, grid_size
    )
    
    return feat_map

def save_checkpoint(epoch, backbone, optimizer, output_dir="./checkpoints"):
    os.makedirs(output_dir, exist_ok=True)
    
    path = os.path.join(output_dir, f"self_supervised_epoch_{epoch}.pth")
    torch.save({
        "epoch": epoch,
        "backbone": backbone.state_dict(),
        "optimizer": optimizer.state_dict(),
    }, path)
    
    backbone_only_path = os.path.join(output_dir, f"backbone_only_epoch_{epoch}.pth")
    torch.save(backbone.state_dict(), backbone_only_path)
    
    print(f"Checkpoint saved:")
    print(f"  Full: {path}")
    print(f"  Backbone: {backbone_only_path}")

print("Helper functions defined")

## Training Loop

In [ ]:
print(f"Starting self-supervised fine-tuning...\n")

for epoch in range(EPOCHS):
    student_model.train()
    teacher_model.eval()
    
    total_loss = 0.0
    momentum_loss_sum = 0.0
    spatial_loss_sum = 0.0
    
    pbar = tqdm(loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    
    for batch_idx, (img_weak, img_strong) in enumerate(pbar):
        img_weak = img_weak.to(DEVICE)
        img_strong = img_strong.to(DEVICE)
        
        with torch.no_grad():
            teacher_feat_tokens = teacher_model.forward_features(img_weak)
        
        student_feat_tokens = student_model.forward_features(img_strong)
        
        student_feat_map = extract_patch_features(student_feat_tokens)
        teacher_feat_map = extract_patch_features(teacher_feat_tokens)
        
        momentum_loss_val = momentum_loss(
            student_feat_tokens[:, 1:, :].mean(dim=1),
            teacher_feat_tokens[:, 1:, :].mean(dim=1)
        )
        
        spatial_loss_val = spatial_loss(student_feat_map)
        
        total_loss_val = MOMENTUM_WEIGHT * momentum_loss_val + SPATIAL_WEIGHT * spatial_loss_val
        
        optimizer.zero_grad()
        total_loss_val.backward()
        optimizer.step()
        
        update_teacher(student_model, teacher_model, tau=MOMENTUM)
        
        total_loss += total_loss_val.item()
        momentum_loss_sum += momentum_loss_val.item()
        spatial_loss_sum += spatial_loss_val.item()
        
        pbar.set_postfix({
            "loss": f"{total_loss_val.item():.4f}",
            "mom": f"{momentum_loss_val.item():.4f}",
            "spa": f"{spatial_loss_val.item():.4f}"
        })
    
    avg_loss = total_loss / len(loader)
    avg_momentum_loss = momentum_loss_sum / len(loader)
    avg_spatial_loss = spatial_loss_sum / len(loader)
    
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print(f"  Total Loss: {avg_loss:.4f}")
    print(f"  Momentum Loss: {avg_momentum_loss:.4f}")
    print(f"  Spatial Loss: {avg_spatial_loss:.4f}")
    
    if (epoch + 1) % 5 == 0 or epoch == 0:
        save_checkpoint(epoch+1, student_model, optimizer)

print("\nSelf-supervised fine-tuning completed!")

## Save Final Models

In [ ]:
os.makedirs("./checkpoints", exist_ok=True)

final_path = "./checkpoints/self_supervised_final.pth"
torch.save({
    "epoch": EPOCHS,
    "backbone": student_model.state_dict(),
    "optimizer": optimizer.state_dict(),
}, final_path)

final_backbone_path = "./checkpoints/self_supervised_backbone_final.pth"
torch.save(student_model.state_dict(), final_backbone_path)

print("Final checkpoints saved:")
print(f"  Full checkpoint: {final_path}")
print(f"  Backbone only: {final_backbone_path}")
print(f"\nTo use with GSNet, set RSIB_CKPT to: {os.path.abspath(final_backbone_path)}")

## Summary

In [ ]:
print("="*60)
print("Self-Supervised Fine-Tuning Summary")
print("="*60)
print(f"\nModel: DINOv3 ViT-L/16")
print(f"Dataset: LandDiscover-50K (images only)")
print(f"Total images: {len(dataset)}")
print(f"\nTraining Configuration:")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Epochs: {EPOCHS}")
print(f"  Unfrozen blocks: {NUM_UNFROZEN_BLOCKS}")
print(f"  Trainable params: {trainable_params:,} ({trainable_pct:.1f}%)")
print(f"\nLoss Functions:")
print(f"  Momentum Consistency (weight={MOMENTUM_WEIGHT})")
print(f"  Spatial Patch Contrastive (weight={SPATIAL_WEIGHT})")
print(f"\nHyperparameters:")
print(f"  Momentum coefficient: {MOMENTUM}")
print(f"  Temperature: {TEMPERATURE}")
print(f"  Backbone LR: {backbone_lr}")
print(f"\nCheckpoints saved in: ./checkpoints/")
print(f"\nNext step: Train GSNet with these backbones and evaluate mIoU")
print("="*60)